# Bear Attack Analysis: Wyoming vs Poland
## Comparing Attack Rates to Make an Informed Decision

Our hypothesis: Smaller bear density could mean safer hiking, but we'll test this with real data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set up plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Step 1: Load the aggregated brown bear data

In [ ]:
# Load the global brown bear attacks dataset
brown_bears = pd.read_csv('../mnt/user-data/uploads/global_brownbear_attacks.csv')

# Display first few rows and info
print("Brown Bear Dataset:")
print(brown_bears.head())
print("\nDataset Info:")
print(brown_bears.info())

## Step 2: Extract and compare Wyoming vs Poland basic metrics

In [ ]:
# Extract Wyoming and Poland data
wyoming = brown_bears[brown_bears['Country/State'] == 'Wyoming'].iloc[0]
poland = brown_bears[brown_bears['Country/State'] == 'Poland'].iloc[0]

print("=" * 60)
print("WYOMING DATA (2000-2015)")
print("=" * 60)
print(f"Number of Bears: {wyoming['NumberofBrownBears']}")
print(f"Number of Attacks: {wyoming['NumberofAttacks(2000–2015)']}")
print(f"Number of Fatalities: {wyoming['NumberofFatalities(2000–2015)']}")
print(f"Bear Range (km²): {wyoming['BrownBearRange(km2)']}")
print(f"Human Density (people/km²): {wyoming['HumanDensity(inhabitants/km2)']}")
print(f"Bear Density (bears/1000km²): {wyoming['BrownBearDensity(bears/1000km2)']}")

print("\n" + "=" * 60)
print("POLAND DATA (2000-2015)")
print("=" * 60)
print(f"Number of Bears: {poland['NumberofBrownBears']}")
print(f"Number of Attacks: {poland['NumberofAttacks(2000–2015)']}")
print(f"Number of Fatalities: {poland['NumberofFatalities(2000–2015)']}")
print(f"Bear Range (km²): {poland['BrownBearRange(km2)']}")
print(f"Human Density (people/km²): {poland['HumanDensity(inhabitants/km2)']}")
print(f"Bear Density (bears/1000km²): {poland['BrownBearDensity(bears/1000km2)']}")

## Step 3: Calculate key metrics - Attack rates and fatality rates

In [ ]:
# Function to calculate metrics for a region
def calculate_metrics(region_data):
    num_bears = region_data['NumberofBrownBears']
    num_attacks = region_data['NumberofAttacks(2000–2015)']
    num_fatalities = region_data['NumberofFatalities(2000–2015)']
    
    # Calculate rates
    attack_per_bear = (num_attacks / num_bears) * 100 if num_bears > 0 else 0
    fatality_rate = (num_fatalities / num_attacks) * 100 if num_attacks > 0 else 0
    fatality_per_bear = (num_fatalities / num_bears) * 100 if num_bears > 0 else 0
    
    return {
        'num_bears': num_bears,
        'num_attacks': num_attacks,
        'num_fatalities': num_fatalities,
        'attacks_per_bear_percent': attack_per_bear,
        'fatality_rate_percent': fatality_rate,
        'fatalities_per_bear_percent': fatality_per_bear
    }

# Calculate for both regions
wyoming_metrics = calculate_metrics(wyoming)
poland_metrics = calculate_metrics(poland)

# Create comparison table
comparison_df = pd.DataFrame({
    'Metric': [
        'Number of Bears',
        'Number of Attacks (2000-2015)',
        'Number of Fatalities',
        'Attacks per Bear (%)',
        'Fatality Rate (% of attacks)',
        'Fatalities per Bear (%)'
    ],
    'Wyoming': [
        wyoming_metrics['num_bears'],
        wyoming_metrics['num_attacks'],
        wyoming_metrics['num_fatalities'],
        f"{wyoming_metrics['attacks_per_bear_percent']:.3f}%",
        f"{wyoming_metrics['fatality_rate_percent']:.1f}%",
        f"{wyoming_metrics['fatalities_per_bear_percent']:.3f}%"
    ],
    'Poland': [
        poland_metrics['num_bears'],
        poland_metrics['num_attacks'],
        poland_metrics['num_fatalities'],
        f"{poland_metrics['attacks_per_bear_percent']:.3f}%",
        f"{poland_metrics['fatality_rate_percent']:.1f}%",
        f"{poland_metrics['fatalities_per_bear_percent']:.3f}%"
    ]
})

print("\n" + "=" * 80)
print("COMPARISON: ATTACK RATES AND FATALITY RATES")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("\n⚠️  NOTE: Poland's sample size is much smaller, so results may be less reliable")

## Step 4: Load individual attack records and prepare for wild attack filtering

In [ ]:
# Load the individual attack records
attacks = pd.read_csv('../mnt/user-data/uploads/data__3_.csv')

print("Individual Attacks Dataset:")
print(f"Total records: {len(attacks)}")
print(f"\nColumns: {attacks.columns.tolist()}")
print(f"\nFirst few records:")
print(attacks.head(3))

## Step 5: Filter for wild attacks only in Wyoming and Poland

In [ ]:
# Check what's in the Type column
print("Attack Types available:")
print(attacks['Type'].unique())
print(f"\nCount by Type:")
print(attacks['Type'].value_counts())

In [ ]:
# Filter for wild attacks only
wild_attacks = attacks[attacks['Type'] == 'Wild'].copy()

print(f"Total wild attacks: {len(wild_attacks)}")
print(f"\nBear types in wild attacks:")
print(wild_attacks['Type of bear'].value_counts())

In [ ]:
# Now filter for Wyoming and Poland specifically
# Check what location strings contain Wyoming and Poland
print("Sample locations in dataset:")
print(wild_attacks['Location'].unique()[:20])

In [ ]:
# Filter for Wyoming attacks
wyoming_attacks = wild_attacks[wild_attacks['Location'].str.contains('Wyoming', case=False, na=False)]

# Filter for Poland attacks  
poland_attacks = wild_attacks[wild_attacks['Location'].str.contains('Poland', case=False, na=False)]

print(f"Wild attacks in Wyoming: {len(wyoming_attacks)}")
print(f"Wild attacks in Poland: {len(poland_attacks)}")

print("\n" + "="*60)
print("WYOMING WILD ATTACKS:")
print("="*60)
if len(wyoming_attacks) > 0:
    print(wyoming_attacks[['Name', 'Date', 'Type of bear', 'Location', 'Description']].to_string())
else:
    print("No Wyoming attacks found")

print("\n" + "="*60)
print("POLAND WILD ATTACKS:")
print("="*60)
if len(poland_attacks) > 0:
    print(poland_attacks[['Name', 'Date', 'Type of bear', 'Location', 'Description']].to_string())
else:
    print("No Poland attacks found")

## Step 6: Summary and Insights

In [ ]:
print("\n" + "="*80)
print("SUMMARY: WILD ATTACKS ONLY")
print("="*80)
print(f"Wyoming wild attacks: {len(wyoming_attacks)} out of {wyoming_metrics['num_attacks']} total")
print(f"Poland wild attacks: {len(poland_attacks)} out of {poland_metrics['num_attacks']} total")
print("\nNote: Some attacks may not have explicit location names, so these counts may be underestimates.")